# 03. 메시지(Messages) — LangGraph 대화의 기본 단위

> LangGraph 상태의 핵심은 결국 **메시지 리스트**예요. System/Human/AI/Tool 4가지 타입과 메타데이터, `add_messages` 리듀서의 동작을 명확히 이해하고 갑니다.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `SystemMessage`, `HumanMessage`, `AIMessage`, `ToolMessage` 4가지 메시지 타입의 역할과 사용 시점을 설명할 수 있어요
2. `name`, `id` 메타데이터를 활용하여 다중 사용자 환경에서 메시지를 추적할 수 있어요
3. 딕셔너리 형식(`{"role": ..., "content": ...}`)으로 메시지를 구성할 수 있어요
4. `ToolMessage`의 `tool_call_id` 매칭 규칙을 이해하고 도구 호출 전체 흐름을 구현할 수 있어요
5. Provider 네이티브 형식과 표준 `content_blocks`를 활용하여 멀티모달 메시지를 구성할 수 있어요

## 사전 지식

- 이전 노트북: `02-Models.ipynb` — `init_chat_model`, `invoke/stream/batch`, Tool Calling 기초
- LangChain V0의 메시지 기본 개념

## 메시지란 무엇인가요?

메시지(Message)는 LangChain에서 LLM과 주고받는 대화의 기본 단위예요. 에이전트의 모든 상호작용은 메시지 리스트로 구성되고, LangGraph의 상태(State)에 저장돼요.

각 메시지는 다음 세 가지 핵심 요소로 구성돼요:

| 요소 | 역할 | 예시 |
|------|------|------|
| **역할(Role)** | 메시지 발신자 구분 | `system`, `user`, `assistant`, `tool` |
| **콘텐츠(Content)** | 실제 메시지 내용 | 텍스트, 이미지, 오디오 등 |
| **메타데이터(Metadata)** | 추가 정보 | `name`, `id`, 토큰 사용량, 도구 호출 정보 |

```mermaid
flowchart TD
    A["사용자 입력"]:::input --> B["HumanMessage"]:::input
    C["시스템 설정"]:::process --> D["SystemMessage"]:::process
    B --> E["LLM 모델"]:::process
    D --> E
    E --> F["AIMessage"]:::output
    F --> G{"도구 호출?"}:::process
    G -->|"Yes: tool_calls"| H["도구 실행"]:::storage
    H --> I["ToolMessage"]:::storage
    I --> E
    G -->|"No: content"| J["최종 응답"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

### LangGraph에서 메시지의 위치

LangGraph 에이전트의 상태에는 항상 `messages` 필드가 있어요. 각 노드(Node)는 이 리스트에서 메시지를 읽고 새로운 메시지를 추가해요.

> 🔑 **핵심 개념**: LangGraph에서 `add_messages` reducer는 메시지 리스트를 자동으로 관리해요. 같은 `id`를 가진 메시지는 업데이트되고, 새로운 메시지는 추가돼요. 다음 노트북 `04-StateGraph-Basics.ipynb`에서 자세히 다룰 거예요.

## 환경 설정

In [1]:
# ---------------------------------------------------
# 환경 설정
# ---------------------------------------------------
# dotenv: .env 파일에서 API 키를 로드해요
from dotenv import load_dotenv
import os

# .env 파일의 환경 변수를 로드해요
load_dotenv(override=True)

# LangSmith 추적 설정 (선택 사항)
# LangSmith에서 메시지 흐름을 시각적으로 디버깅할 수 있어요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-Part03-Messages"

True

## 1. 메시지 기본 사용법

메시지 객체를 생성하고 모델에 리스트로 전달하는 가장 기본적인 패턴이에요. 메시지 리스트를 통해 대화 컨텍스트를 유지할 수 있어요.

> 🎯 **강의 포인트**: 이전 노트북에서 `("system", "...")` 튜플 형식을 배웠어요. 이번 노트북에서는 메시지 **객체**를 직접 다뤄요. 두 방식이 동일하게 동작하지만, LangGraph 상태에서는 반드시 객체 형식을 사용해야 해요.

> 💡 **실무 팁**: 튜플 형식 `("human", "...")`, 딕셔너리 형식 `{"role": "user", "content": "..."}`, 객체 형식 `HumanMessage("...")`은 모두 모델에 전달할 때 동일하게 동작해요. 하지만 LangGraph State에서는 **메시지 객체**만 사용할 수 있어요. 세 가지 형식을 음식 주문에 비유하면: 튜플은 "메뉴판 번호로 주문", 딕셔너리는 "종이에 적어서 전달", 객체는 "정식 주문서 작성"이에요. LangGraph는 정식 주문서(객체)만 접수해요.

### 메시지 Import 경로 (V1)

LangChain V1에서는 모든 메시지를 `langchain.messages`에서 가져와요.

| V1 (올바른 방법) | 이전 방식 (피해야 할 방법) |
|-----------------|---------------------------|
| `from langchain.messages import HumanMessage` | `from langchain_core.messages import HumanMessage` |

In [2]:
# ---------------------------------------------------
# V1 메시지 Import 및 모델 초기화
# ---------------------------------------------------
# V1 통합 경로: langchain.messages (langchain_core 대신)
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain.chat_models import init_chat_model

# 기본 모델: gpt-4o-mini (비용 효율, 학생 접근성)
# 다른 모델: "anthropic:claude-sonnet-4-5", "google_genai:gemini-2.0-flash"
model = init_chat_model("openai:gpt-4o-mini")

print(f"모델 초기화 완료: {type(model).__name__}")

모델 초기화 완료: ChatOpenAI


In [3]:
# ---------------------------------------------------
# 메시지 객체를 사용한 기본 모델 호출
# ---------------------------------------------------
# SystemMessage: 모델의 역할과 동작 방식을 지시해요
system_msg = SystemMessage("당신은 친절한 AI 어시스턴트입니다. 한국어로 답변해주세요.")

# HumanMessage: 사용자의 질문이나 요청을 담아요
human_msg = HumanMessage("안녕하세요. LangGraph란 무엇인가요?")

# 메시지 리스트로 모델 호출
messages = [system_msg, human_msg]
response = model.invoke(messages)  # AIMessage 객체를 반환해요

# 응답 확인
print(f"응답 타입: {type(response).__name__}")
print(f"응답 내용: {response.content}")

응답 타입: AIMessage
응답 내용: 안녕하세요! LangGraph는 자연어 처리(NLP)와 관련된 기술 또는 플랫폼일 가능성이 높습니다. 일반적으로 이러한 시스템은 언어 이해, 대화 생성 또는 문서 분석 등 다양한 언어 기반 작업을 수행하는 데 사용됩니다. 하지만 "LangGraph"에 대한 구체적인 정보는 제가 가지고 있는 데이터 기준인 2023년 10월까지는 확인할 수 없습니다. 

이와 관련된 특정한 제품이나 프로젝트에 대한 정보를 찾고 있다면, 더 많은 맥락이나 배경을 제공해 주시면 더욱 도움이 될 수 있습니다. 또는 최신 정보를 확인하기 위해 인터넷 검색을 해보시는 것도 좋은 방법입니다. 도움이 필요하면 언제든지 말씀해 주세요!


### 딕셔너리 형식으로 메시지 구성하기

메시지 객체 대신 딕셔너리로도 메시지를 지정할 수 있어요. `role` 키에는 역할을, `content` 키에는 내용을 넣어요. JSON 직렬화가 필요하거나 외부 API 연동 시 편리해요.

> 💡 **실무 팁**: REST API로 메시지를 받아서 모델에 전달할 때 딕셔너리 형식이 더 편리해요. 하지만 LangGraph 그래프 내부의 상태(State) 저장 시에는 메시지 객체가 필요해요. 두 형식이 자동으로 변환되지 않는다는 점을 기억하세요.

| 딕셔너리 `role` | 메시지 객체 |
|---|---|
| `system` | `SystemMessage` |
| `user` | `HumanMessage` |
| `assistant` | `AIMessage` |
| `tool` | `ToolMessage` |


In [4]:
# ---------------------------------------------------
# 딕셔너리 형식으로 메시지 구성
# ---------------------------------------------------
# 딕셔너리의 role 키: "system", "user", "assistant" 사용
# ("human" 대신 "user"를 사용해야 해요)
messages_dict = [
    {"role": "system", "content": "당신은 친절한 어시스턴트입니다."},
    {"role": "user", "content": "대한민국의 수도는?"},
    {"role": "assistant", "content": "대한민국의 수도는 서울입니다."},
    {"role": "user", "content": "영어로 작성해줘."},
]

# 딕셔너리 형식과 객체 형식은 동일하게 동작해요
response_dict = model.invoke(messages_dict)
print(f"응답: {response_dict.content}")
print()

응답: The capital of South Korea is Seoul.



## 2. 4가지 메시지 타입 상세 가이드

LangChain V1은 4가지 핵심 메시지 타입을 제공해요. 각 타입은 대화에서 명확한 역할을 담당해요.

| 메시지 타입 | 역할 | 대화에서 위치 |
|------------|------|---------------|
| `SystemMessage` | 모델 동작 방식과 역할 정의 | 대화 시작 부분 |
| `HumanMessage` | 사용자 입력 (텍스트 + 멀티모달) | 사용자 차례마다 |
| `AIMessage` | 모델 생성 응답 (텍스트 + 도구 호출) | 모델 응답마다 |
| `ToolMessage` | 도구 실행 결과 전달 | 도구 호출 직후 |

### 2-1. SystemMessage — 모델 행동 지침

`SystemMessage`는 모델의 페르소나, 응답 스타일, 제약 조건 등을 정의해요. 효과적인 시스템 메시지는 모델이 일관된 방식으로 동작하도록 만들어요.

> 🔑 **핵심 개념**: 시스템 메시지는 대화 전체에 영향을 미치는 "설정" 이에요. 역할(`role`), 제약(`constraint`), 형식(`format`) 세 가지 요소를 포함하는 것이 좋아요.

In [5]:
# ---------------------------------------------------
# SystemMessage: 모델 역할과 동작 방식 지정
# ---------------------------------------------------
from langchain.messages import SystemMessage, HumanMessage

# 역할(role) + 제약(constraint) + 형식(format)을 포함한 시스템 메시지
system_msg = SystemMessage(
    "You are a helpful Python coding assistant. "
    "Always provide working code examples. "
    "Use Korean for explanations but English for code."
)

# 시스템 메시지를 포함한 대화 구성
messages = [
    system_msg,
    HumanMessage("리스트에서 중복을 제거하는 코드를 보여줘."),
]

response = model.invoke(messages)
print(response.content)

리스트에서 중복을 제거하는 방법은 여러 가지가 있지만, 가장 일반적인 방법은 `set`을 사용하는 것입니다. `set`은 중복을 허용하지 않기 때문에 리스트를 집합으로 변환한 다음 다시 리스트로 변환할 수 있습니다. 아래는 그 예시입니다.

```python
# 중복된 요소가 있는 리스트
my_list = [1, 2, 2, 3, 4, 4, 5]

# 중복 제거
unique_list = list(set(my_list))

# 결과 출력
print(unique_list)
```

이 코드를 실행하면 결과로 중복이 제거된 리스트인 `[1, 2, 3, 4, 5]`가 출력됩니다. 

주의할 점은 `set`으로 변환하면 원래 리스트의 순서가 보존되지 않기 때문에, 순서를 유지하면서 중복을 제거하고 싶다면 다음과 같은 방법을 사용할 수 있습니다.

```python
# 중복된 요소가 있는 리스트
my_list = [1, 2, 2, 3, 4, 4, 5]

# 순서를 유지하며 중복 제거
unique_list = []
for item in my_list:
    if item not in unique_list:
        unique_list.append(item)

# 결과 출력
print(unique_list)
```

이 방법을 사용하면 출력 결과는 여전히 `[1, 2, 3, 4, 5]`가 되지만, 요소의 순서는 원래 리스트와 동일하게 유지됩니다.


### 2-2. HumanMessage — 사용자 입력

`HumanMessage`는 사용자가 보내는 모든 입력을 담아요. 텍스트뿐 아니라 이미지, 오디오 등 멀티모달 콘텐츠도 포함할 수 있어요.

> 🎯 **강의 포인트**: `name`과 `id` 메타데이터는 LangGraph의 `add_messages` reducer와 함께 매우 중요해요. 같은 `id`를 가진 메시지는 업데이트(덮어쓰기)되고, `name`은 다중 에이전트 환경에서 발신자를 구별하는 데 사용해요.

In [6]:
# ---------------------------------------------------
# HumanMessage: 기본 생성 및 메타데이터 추가
# ---------------------------------------------------
from langchain.messages import HumanMessage

# 기본 텍스트 메시지
basic_human = HumanMessage("안녕하세요?")
# 기본 HumanMessage:
print(f"  content: {basic_human.content}")
print(f"  type: {basic_human.type}")
print()

# 메타데이터를 포함한 메시지
# name: 다중 사용자 환경에서 발신자를 식별해요
# id: 메시지를 고유하게 식별해요 (add_messages에서 업데이트 기준)
human_with_meta = HumanMessage(
    content="LangGraph가 궁금해요!",
    name="student_alex",  # 발신자 이름 (선택 사항)
    id="msg-001",          # 고유 식별자 (선택 사항)
)

# 메타데이터 포함 HumanMessage:
print(f"  content: {human_with_meta.content}")
print(f"  name: {human_with_meta.name}")
print(f"  id: {human_with_meta.id}")

  content: 안녕하세요?
  type: human

  content: LangGraph가 궁금해요!
  name: student_alex
  id: msg-001


In [7]:
# ---------------------------------------------------
# 메타데이터가 포함된 메시지로 모델 호출
# ---------------------------------------------------
# name 필드는 모델에게 발신자 정보를 전달해요
response = model.invoke([human_with_meta])
print(f"응답: {response.content}")

# 반환된 AIMessage 구조 확인
print()
print(f"응답 타입: {type(response).__name__}")
print(f"응답 id: {response.id}")

응답: LangGraph는 자연어 처리(NLP)와 관련된 다양한 태스크를 지원하는 언어 모델 및 그래프 기반 시스템을 의미할 수 있습니다. 'LangGraph'가 특정한 프로젝트나 시스템을 지칭하는 경우, 추가적인 컨텍스트가 필요할 수 있습니다. 일반적으로 언어 모델은 텍스트 데이터를 이해하고 생성하는 데 사용되며, 그래프는 데이터 간의 관계를 표현하는 좋은 방법입니다.

LangGraph의 주된 기능이나 사용 사례에 대해 더 알고 싶으신가요? 예를 들어, 특정 애플리케이션, 연구 분야, 또는 사용 방법에 대해 질문하실 수 있습니다!

응답 타입: AIMessage
응답 id: lc_run--019dd1ee-4049-7bb3-8890-5aca8703e10c-0


### 2-3. AIMessage — 모델 응답

`AIMessage`는 모델이 생성한 응답이에요. 텍스트 응답 외에도 도구 호출(`tool_calls`) 정보, 토큰 사용량(`usage_metadata`), 제공자 메타데이터를 포함해요.

> ⚠️ **자주 하는 실수**: 도구 호출이 발생하면 `response.content`가 빈 문자열이에요! `response.tool_calls`에 호출 정보가 들어있어요. 도구 호출 여부를 먼저 확인하고 처리해야 해요.

In [8]:
# ---------------------------------------------------
# AIMessage 구조 상세 분석
# ---------------------------------------------------
from langchain.messages import HumanMessage

# 일반 텍스트 응답
response = model.invoke([HumanMessage("파이썬이란?")])

# === AIMessage 구조 ===
print(f"[type]           : {response.type}")
print(f"[id]             : {response.id}")
print(f"[content]        : {response.content[:80]}...")  # 너무 길면 일부만 출력
print(f"[tool_calls]     : {response.tool_calls}")  # 도구 호출 없으면 빈 리스트
print(f"[usage_metadata] : {response.usage_metadata}")
print(f"[model_name]     : {response.response_metadata.get('model_name', 'N/A')}")

[type]           : ai
[id]             : lc_run--019dd1ee-50e8-7141-aa8f-977b38ca0246-0
[content]        : 파이썬(Python)은 1991년에 귀도 반 로썸(Guido van Rossum)이 개발한 고급 프로그래밍 언어입니다. 파이썬은 코드의 가독성이...
[tool_calls]     : []
[usage_metadata] : {'input_tokens': 14, 'output_tokens': 352, 'total_tokens': 366, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
[model_name]     : gpt-4o-mini-2024-07-18


In [9]:
# ---------------------------------------------------
# 다중 턴 대화: AIMessage를 이전 대화에 포함시키기
# ---------------------------------------------------
# 대화 이력을 유지하려면 이전 AIMessage를 메시지 리스트에 포함시켜요
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# 3번의 대화 이력이 있는 상황을 구성해요
conversation_history = [
    SystemMessage("당신은 친절한 어시스턴트입니다."),
    HumanMessage("대한민국의 수도는?"),
    AIMessage("대한민국의 수도는 서울입니다."),  # 이전 응답을 포함시켜요
    HumanMessage("그 도시의 인구는?"),           # 이전 문맥을 참고해야 해요
]

# 모델이 이전 대화에서 "서울"임을 파악해서 답변해요
response = model.invoke(conversation_history)
print(f"응답: {response.content}")
print()
# 모델이 이전 대화 맥락을 기억하여 서울의 인구를 답변했어요!

응답: 서울의 인구는 약 9백만 명 이상입니다. 하지만 인구는 시간에 따라 변동이 있을 수 있으므로, 최신 통계를 확인하는 것이 좋습니다. 2023년 현재의 정확한 인구 수를 확인하고 싶다면 서울시청이나 통계청의 공식 웹사이트를 참고해 보세요.



## 3. ToolMessage — 도구 호출 흐름

`ToolMessage`는 도구 실행 결과를 모델에 다시 전달하는 특별한 메시지예요. LangGraph 에이전트의 ReAct 루프에서 핵심 역할을 해요.

도구 호출 전체 흐름은 아래와 같이 4단계로 이루어져요:

```mermaid
sequenceDiagram
    participant U as 사용자
    participant M as LLM 모델
    participant T as 도구 실행

    U->>M: HumanMessage (서울 날씨 알려줘)
    M-->>T: AIMessage (tool_calls: get_weather, id=call_123)
    T-->>M: ToolMessage (content: 맑음 10도, tool_call_id=call_123)
    M-->>U: AIMessage (content: 서울은 현재 맑고 10도입니다)
```

### AIMessage의 두 가지 모드

| 모드 | `content` | `tool_calls` | 의미 |
|------|-----------|-------------|------|
| 텍스트 응답 | `"서울의 수도는..."` | `[]` (빈 리스트) | LLM이 직접 답변 |
| 도구 호출 | `""` (빈 문자열) | `[{name, args, id}]` | LLM이 도구 사용을 요청 |

> 🔑 **핵심 개념**: `ToolMessage`의 `tool_call_id`는 반드시 해당 도구를 요청한 `AIMessage`의 `tool_calls[i]['id']`와 일치해야 해요. 이 ID를 통해 모델은 어떤 도구 호출에 대한 결과인지 파악해요.

> ⚠️ **자주 하는 실수**: `tool_call_id`를 잘못 입력하거나 누락하면 모델이 어느 도구 호출에 대한 응답인지 알 수 없어서 오류가 발생해요. ID 매칭이 핵심이에요!

In [10]:
# ---------------------------------------------------
# ToolMessage 수동 구성: tool_call_id 매칭 확인
# ---------------------------------------------------
from langchain.messages import AIMessage, HumanMessage, ToolMessage

# 1단계: 모델이 도구 호출을 포함한 AIMessage 생성 (수동 구성)
# 실제 LangGraph에서는 모델이 자동으로 생성해요
ai_with_tool_call = AIMessage(
    content="",  # 도구 호출 시 content는 빈 문자열이에요
    tool_calls=[
        {
            "name": "get_weather",            # 호출할 도구 이름
            "args": {"location": "서울"},     # 도구에 전달할 인자
            "id": "call_abc123",              # 고유 호출 ID (ToolMessage와 매칭 필수!)
        }
    ],
)

# AIMessage (도구 호출 요청):
print(f"  content: '{ai_with_tool_call.content}'  <- 도구 호출 시 비어 있어요")
print(f"  tool_calls: {ai_with_tool_call.tool_calls}")

  content: ''  <- 도구 호출 시 비어 있어요
  tool_calls: [{'name': 'get_weather', 'args': {'location': '서울'}, 'id': 'call_abc123', 'type': 'tool_call'}]


In [11]:
# ---------------------------------------------------
# 2단계: 도구 실행 결과를 ToolMessage로 래핑
# ---------------------------------------------------
# 도구 실행 결과 (실제로는 외부 API나 함수가 반환해요)
weather_result = "날씨: 맑음, 기온: 10도C, 습도: 45%"

# ToolMessage 생성
# tool_call_id는 AIMessage의 tool_calls[0]['id']와 반드시 일치해야 해요!
tool_msg = ToolMessage(
    content=weather_result,
    tool_call_id="call_abc123",  # AIMessage의 id와 정확히 동일해야 해요
)

# ToolMessage (도구 실행 결과):
print(f"  content: {tool_msg.content}")
print(f"  tool_call_id: {tool_msg.tool_call_id}")
print(f"  type: {tool_msg.type}")

  content: 날씨: 맑음, 기온: 10도C, 습도: 45%
  tool_call_id: call_abc123
  type: tool


In [12]:
# ---------------------------------------------------
# 3단계: 전체 도구 호출 대화 흐름 실행
# ---------------------------------------------------
# 완전한 대화 흐름 구성:
# HumanMessage -> AIMessage(tool_calls) -> ToolMessage -> AIMessage(응답)
full_conversation = [
    HumanMessage("서울의 현재 날씨는 어때?"),   # 1단계: 사용자 질문
    ai_with_tool_call,                           # 2단계: 모델이 도구 호출 결정
    tool_msg,                                    # 3단계: 도구 실행 결과 전달
]

# 4단계: 모델이 도구 결과를 보고 최종 응답 생성
final_response = model.invoke(full_conversation)
# 최종 응답:
print(final_response.content)

현재 서울의 날씨는 맑고, 기온은 10도C, 습도는 45%입니다.


### 실제 도구 바인딩과 자동 도구 호출 확인

수동으로 메시지를 구성하는 방법을 이해했어요. 이제 실제로 모델에 도구를 바인딩하고, 모델이 자동으로 도구를 선택하는 과정을 확인해볼게요.

In [13]:
# ---------------------------------------------------
# 실제 도구 바인딩 후 AIMessage 구조 확인
# ---------------------------------------------------
from pydantic import BaseModel, Field
from langchain.messages import HumanMessage


# Pydantic으로 도구 스키마 정의
class GetWeather(BaseModel):
    """현재 날씨 정보를 가져오는 도구예요."""

    location: str = Field(description="도시명. 예: 서울, 부산")


class SearchNews(BaseModel):
    """뉴스를 검색하는 도구예요."""

    query: str = Field(description="검색어")
    max_results: int = Field(default=3, description="최대 결과 수")


# 모델에 도구를 바인딩해요
model_with_tools = model.bind_tools([GetWeather, SearchNews])

# 날씨 관련 질문: 모델이 GetWeather 도구를 선택해요
response_tool = model_with_tools.invoke([HumanMessage("오늘 부산 날씨 알려줘")])

# === 도구 호출 발생 시 AIMessage ===
print(f"content (비어 있어요): '{response_tool.content}'")
print()
# tool_calls 내용:
for tc in response_tool.tool_calls:
    print(f"  이름: {tc['name']}")
    print(f"  인자: {tc['args']}")
    print(f"  ID:  {tc['id']}")
    #   (이 ID를 ToolMessage의 tool_call_id에 사용해야 해요)

content (비어 있어요): ''

  이름: GetWeather
  인자: {'location': '부산'}
  ID:  call_KOFB6OAPqvWZ4FK9xuXcuXu8


In [14]:
# ---------------------------------------------------
# tool_call_id 매칭으로 완전한 ReAct 루프 구성
# ---------------------------------------------------
# 실제 도구를 실행하는 함수 (여기서는 더미 데이터 반환)
def execute_get_weather(location: str) -> str:
    """날씨 API를 호출하는 더미 함수예요."""
    weather_db = {
        "서울": "맑음, 15도C",
        "부산": "흐림, 18도C",
        "제주": "비, 20도C",
    }
    return weather_db.get(location, f"{location}: 데이터 없음")


# tool_call_id를 AIMessage에서 직접 추출해요
tool_call = response_tool.tool_calls[0]  # 첫 번째 도구 호출 정보
location_arg = tool_call["args"]["location"]   # 도구 인자 추출
call_id = tool_call["id"]                       # ID 추출 (매칭 핵심!)

# 도구 실행 및 ToolMessage 생성
weather_data = execute_get_weather(location_arg)
auto_tool_msg = ToolMessage(
    content=weather_data,
    tool_call_id=call_id,  # AIMessage의 ID와 정확히 매칭!
)

# 전체 ReAct 대화 흐름 실행
original_question = HumanMessage("오늘 부산 날씨 알려줘")
final_answer = model_with_tools.invoke(
    [original_question, response_tool, auto_tool_msg]
)

# === 완전한 ReAct 루프 결과 ===
print(f"질문: {original_question.content}")
print(f"도구 실행 결과: {weather_data}")
print(f"최종 응답: {final_answer.content}")

질문: 오늘 부산 날씨 알려줘
도구 실행 결과: 흐림, 18도C
최종 응답: 오늘 부산의 날씨는 흐림이며, 기온은 18도C입니다.


## 4. 메시지 콘텐츠 형식 — 텍스트부터 멀티모달까지

메시지의 `content`는 단순 문자열부터 복잡한 멀티모달 블록까지 다양한 형식을 지원해요. LangChain V1에서는 두 가지 멀티모달 형식을 제공해요.

| 형식 | 특징 | 사용 시점 |
|------|------|----------|
| **문자열** | 단순 텍스트 | 기본 대화, 텍스트 처리 |
| **Provider 네이티브** | `image_url` 형식 | OpenAI, Anthropic 특정 기능 사용 시 |
| **표준 content_blocks** | `url` 형식 (V1 신기능) | 모든 제공자에서 동일하게 동작 |

> ⚠️ **자주 하는 실수**: Provider 네이티브 형식(`image_url`)으로 작성한 코드를 Anthropic이나 Gemini로 교체할 때 형식이 달라서 오류가 발생해요. 표준 `content_blocks`를 사용하면 `init_chat_model`의 모델명만 바꾸면 되므로 **제공자 교체가 코드 한 줄**로 끝나요.

> 💡 **실무 팁**: 새로운 코드에서는 **표준 `content_blocks`** 방식을 사용하세요. 제공자를 변경해도 코드를 수정할 필요가 없어서 유지보수가 편해요. Provider 네이티브 형식은 특정 제공자의 고급 기능이 필요할 때만 사용해요.

In [15]:
# ---------------------------------------------------
# 콘텐츠 형식 1: 문자열 (가장 기본)
# ---------------------------------------------------
from langchain.messages import HumanMessage

# 방법 1: 문자열 콘텐츠 (가장 간단해요)
string_message = HumanMessage("안녕하세요!")
# [문자열 형식]
print(f"  content 타입: {type(string_message.content).__name__}")
print(f"  content: {string_message.content}")
print()

  content 타입: str
  content: 안녕하세요!



In [16]:
# ---------------------------------------------------
# 콘텐츠 형식 2: Provider 네이티브 (image_url 형식)
# ---------------------------------------------------
# OpenAI API 형식을 그대로 사용해요
# 이미지를 base64로 인코딩하여 전달하면 외부 URL 접근 없이 안정적이에요

import urllib.request
import base64

# 샘플 이미지를 다운로드하여 base64로 인코딩해요
# picsum.photos에서 강아지 이미지(id=237)를 가져와요 (200x300px)
_img_url = "https://picsum.photos/id/237/200/300.jpg"
_headers = {"User-Agent": "Mozilla/5.0"}
_req = urllib.request.Request(_img_url, headers=_headers)
with urllib.request.urlopen(_req, timeout=15) as _r:
    _img_data = _r.read()
_b64 = base64.b64encode(_img_data).decode()
# data URL 형식으로 변환해요
SAMPLE_IMAGE_DATA_URL = f"data:image/jpeg;base64,{_b64}"

print(f"이미지 다운로드 완료: {len(_img_data)} bytes → base64 {len(_b64)} chars")

# Provider 네이티브 형식: OpenAI 스타일 (list of dicts)
# 외부 URL 대신 base64 data URL을 사용하면 안정적이에요
provider_native_message = HumanMessage(
    content=[
        {"type": "text", "text": "이 이미지에 무엇이 있나요?"},
        {
            "type": "image_url",      # OpenAI/Anthropic 네이티브 형식
            "image_url": {"url": SAMPLE_IMAGE_DATA_URL},  # base64 data URL 사용
        },
    ]
)

# [Provider 네이티브 형식 (image_url + base64)]
print(f"  content 타입: {type(provider_native_message.content).__name__}")
print(f"  블록 수: {len(provider_native_message.content)}")
print(f"  블록[0] type: {provider_native_message.content[0]['type']}")
print(f"  블록[1] type: {provider_native_message.content[1]['type']}")


이미지 다운로드 완료: 11615 bytes → base64 15488 chars
  content 타입: list
  블록 수: 2
  블록[0] type: text
  블록[1] type: image_url


In [17]:
# ---------------------------------------------------
# 콘텐츠 형식 3: 표준 content_blocks (V1 신기능 - 권장)
# ---------------------------------------------------
# content_blocks: 제공자 독립적인 표준 형식이에요
# image_url 대신 url 키를 사용해요 (더 간결해요)
# base64 data URL도 동일하게 사용할 수 있어요

standard_blocks_message = HumanMessage(
    content_blocks=[  # content_blocks 파라미터 사용 (V1 신기능)
        {"type": "text", "text": "이 이미지를 설명해주세요."},
        {
            "type": "image",               # 표준 형식: "image_url" 대신 "image"
            "url": SAMPLE_IMAGE_DATA_URL,  # base64 data URL 사용 (더 간결해요)
        },
    ]
)

# [표준 content_blocks 형식 (V1 권장)]
print(f"  content_blocks 개수: {len(standard_blocks_message.content_blocks)}")
print(f"  content_blocks: {[{'type': b['type']} for b in standard_blocks_message.content_blocks]}")
print()
# Provider 네이티브 vs 표준 content_blocks 비교:
#   네이티브: {"type": "image_url", "image_url": {"url": "..."}}
#   표준:     {"type": "image", "url": "..."}


  content_blocks 개수: 2
  content_blocks: [{'type': 'text'}, {'type': 'image'}]



In [18]:
# ---------------------------------------------------
# 멀티모달 메시지로 모델 호출 (gpt-4o-mini는 비전 지원)
# ---------------------------------------------------
# gpt-4o-mini: 비전 기능 지원, 이미지 URL 분석 가능

# 이미지 분석 결과 (스트리밍):

# Provider 네이티브 형식으로 호출
for chunk in model.stream([provider_native_message]):
    print(chunk.content, end="", flush=True)
print()  # 마지막 줄바꿈

이 이미지에는 검은색 강아지가 있습니다. 강아지가 바닥에 누워 있는 모습이 보입니다.


In [19]:
# ---------------------------------------------------
# 표준 content_blocks로 동일한 결과 확인
# ---------------------------------------------------
# content_blocks 방식도 동일하게 동작해요
# 표준 content_blocks 방식 결과:
response_blocks = model.invoke([standard_blocks_message])
print(response_blocks.content[:200])  # 앞 200자만 출력

이미지에는 작고 귀여운 검은색 강아지가 나와 있습니다. 강아지는 나무 바닥에 위치하고 있으며, 큰 눈과 부드러운 털을 가지고 있습니다. 모습은 호기심이 가득 차 있고, 주위를 기웃거리는 듯한 느낌을 줍니다. 전체적으로 아기자기하고 사랑스러운 분위기를 자아내고 있습니다.


## 5. 메시지 유틸리티 함수

LangGraph에서 자주 사용하는 메시지 처리 패턴들을 배워봐요. `pretty_print()`는 디버깅 시 메시지 구조를 빠르게 확인할 때 유용해요.

In [20]:
# ---------------------------------------------------
# pretty_print(): 메시지를 보기 좋게 출력해요
# ---------------------------------------------------
from langchain.messages import SystemMessage, HumanMessage, AIMessage

# 다양한 메시지 타입을 포함한 대화 구성
sample_messages = [
    SystemMessage("당신은 Python 전문가입니다."),
    HumanMessage("리스트 컴프리헨션이 뭐야?"),
    AIMessage("리스트 컴프리헨션은 기존 리스트에서 새 리스트를 간결하게 만드는 문법이에요."),
    HumanMessage("예시를 보여줘."),
]

# === pretty_print() 출력 ===
for msg in sample_messages:
    msg.pretty_print()  # 각 메시지를 보기 좋게 출력해요
    print()

================================ System Message ================================

당신은 Python 전문가입니다.

================================ Human Message =================================

리스트 컴프리헨션이 뭐야?

================================== Ai Message ==================================

리스트 컴프리헨션은 기존 리스트에서 새 리스트를 간결하게 만드는 문법이에요.

================================ Human Message =================================

예시를 보여줘.



In [21]:
# ---------------------------------------------------
# 메시지 정보 출력 헬퍼 함수
# ---------------------------------------------------
# display_message_tree 같은 외부 헬퍼 함수 없이 직접 구현해요

def print_message_detail(msg) -> None:
    """메시지의 상세 정보를 구조적으로 출력하는 헬퍼 함수예요.

    Args:
        msg: LangChain 메시지 객체 (HumanMessage, AIMessage 등)
    """
    print(f"[{msg.type.upper()}]")

    # 콘텐츠 출력 (200자 제한)
    content_preview = (
        str(msg.content)[:200]
        if msg.content
        else "(비어 있음)"
    )
    print(f"  Content  : {content_preview}")

    # 메타데이터 출력
    if hasattr(msg, "name") and msg.name:
        print(f"  Name     : {msg.name}")
    if hasattr(msg, "id") and msg.id:
        print(f"  ID       : {msg.id}")

    # 도구 호출 정보 (AIMessage)
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"  Tool Calls ({len(msg.tool_calls)}개):")
        for tc in msg.tool_calls:
            print(f"    - {tc['name']}({tc['args']}) [id: {tc['id']}]")

    # 도구 호출 ID (ToolMessage)
    if hasattr(msg, "tool_call_id") and msg.tool_call_id:
        print(f"  tool_call_id: {msg.tool_call_id}")

    # --------------------------------------------------


# 헬퍼 함수로 각 메시지 분석
from langchain.messages import ToolMessage

test_messages = [
    HumanMessage(content="날씨 알려줘", name="user_alex", id="msg-100"),
    AIMessage(
        content="",
        tool_calls=[{"name": "get_weather", "args": {"location": "서울"}, "id": "call_xyz"}],
    ),
    ToolMessage(content="맑음 15도C", tool_call_id="call_xyz"),
]

# === 메시지 상세 정보 ===
for msg in test_messages:
    print_message_detail(msg)

[HUMAN]
  Content  : 날씨 알려줘
  Name     : user_alex
  ID       : msg-100
[AI]
  Content  : (비어 있음)
  Tool Calls (1개):
    - get_weather({'location': '서울'}) [id: call_xyz]
[TOOL]
  Content  : 맑음 15도C
  tool_call_id: call_xyz


## 6. 실습: 멀티 에이전트 대화 구성

지금까지 배운 개념을 종합해서 직접 대화를 구성해봐요. `name` 필드를 사용하여 두 에이전트가 주고받는 대화를 만들어보세요.

In [ ]:
# ============================================================
# 구현 예시: name 필드를 사용한 두 에이전트 대화 구성
# ============================================================
from langchain.messages import SystemMessage, HumanMessage, AIMessage

multi_agent_conversation = [
    SystemMessage(content="다음은 두 AI 에이전트 간의 협업 대화입니다."),
    HumanMessage(
        content="최근 LangGraph 도입 사례를 조사했습니다. 상태 기반 워크플로우와 HITL 수요가 큽니다.",
        name="researcher",
    ),
    AIMessage(
        content="좋습니다. 이를 바탕으로 교육 커리큘럼에서는 StateGraph, checkpoint, interrupt를 핵심 모듈로 배치하겠습니다.",
        name="analyst",
    ),
    HumanMessage(
        content="그럼 초급 학습자에게 가장 먼저 보여줄 예제는 무엇이 좋을까요?",
        name="researcher",
    ),
]

response = model.invoke(multi_agent_conversation)
print(response.content)


## 7. 메시지 타입 확인과 대화 이력 관리

실무에서는 메시지 타입을 확인하거나 필터링해야 할 경우가 자주 생겨요. 특히 LangGraph 상태(State)에서 메시지를 처리할 때 유용한 패턴들을 알아봐요.

> 💡 **실무 팁**: LangGraph의 `add_messages` reducer는 내부적으로 메시지의 `type` 속성을 사용해요. 커스텀 노드를 작성할 때 메시지 타입을 확인하는 로직이 자주 필요해요.

In [23]:
# ---------------------------------------------------
# 메시지 타입 확인 패턴
# ---------------------------------------------------
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

# 다양한 타입의 메시지 리스트
mixed_messages = [
    SystemMessage("시스템 설정"),
    HumanMessage("질문"),
    AIMessage("답변"),
    ToolMessage(content="도구 결과", tool_call_id="call_1"),
]

# 메시지 타입 분류:
for i, msg in enumerate(mixed_messages):
    # .type 속성으로 메시지 종류를 확인해요
    print(f"  [{i}] type={msg.type!r:12} | class={type(msg).__name__}")

print()

# isinstance()로 특정 타입 필터링
human_messages = [m for m in mixed_messages if isinstance(m, HumanMessage)]
ai_messages = [m for m in mixed_messages if isinstance(m, AIMessage)]

print(f"HumanMessage 수: {len(human_messages)}")
print(f"AIMessage 수:    {len(ai_messages)}")

  [0] type='system'     | class=SystemMessage
  [1] type='human'      | class=HumanMessage
  [2] type='ai'         | class=AIMessage
  [3] type='tool'       | class=ToolMessage

HumanMessage 수: 1
AIMessage 수:    1


In [24]:
# ---------------------------------------------------
# 대화 이력 요약 패턴
# ---------------------------------------------------
# LangGraph의 Conversation Summary 노트북에서 더 자세히 다뤄요
# 여기서는 메시지 구조를 이해하기 위한 기본 패턴이에요

def summarize_conversation(messages: list) -> str:
    """대화 이력을 한 줄로 요약하는 헬퍼 함수예요.

    Args:
        messages: 메시지 객체 리스트

    Returns:
        대화 이력 요약 문자열
    """
    summary_lines = []
    for msg in messages:
        # 각 메시지를 [타입] 내용 형태로 정리해요
        content_preview = str(msg.content)[:50] if msg.content else "(도구 호출)"
        role_label = {
            "system": "[시스템]",
            "human": "[사용자]",
            "ai": "[AI   ]",
            "tool": "[도구  ]",
        }.get(msg.type, f"[{msg.type}]")
        summary_lines.append(f"  {role_label} {content_preview}")
    return "\n".join(summary_lines)


# 실제 대화를 구성하고 요약해봐요
sample_conversation = [
    SystemMessage("당신은 수학 선생님입니다."),
    HumanMessage("1+1은?"),
    AIMessage("1+1은 2예요!"),
    HumanMessage("그럼 2+2는?"),
]

# 대화 이력 요약:
print(summarize_conversation(sample_conversation))

  [시스템] 당신은 수학 선생님입니다.
  [사용자] 1+1은?
  [AI   ] 1+1은 2예요!
  [사용자] 그럼 2+2는?


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **4가지 메시지 타입**: `SystemMessage`(모델 지침), `HumanMessage`(사용자 입력), `AIMessage`(모델 응답), `ToolMessage`(도구 결과) 각각의 역할과 사용 시점
- **메시지 메타데이터**: `name`으로 발신자를 구별하고, `id`로 메시지를 고유 추적해요. LangGraph의 `add_messages` reducer와 연계돼요
- **딕셔너리 형식**: `{"role": "user", "content": "..."}` 형식으로 간결하게 메시지를 구성해요. 역할 이름은 `system`, `user`, `assistant`, `tool`을 사용해요
- **ToolMessage ID 매칭**: `tool_call_id`는 반드시 `AIMessage`의 `tool_calls[i]['id']`와 일치해야 해요. ID 불일치는 가장 흔한 도구 호출 오류예요
- **멀티모달 콘텐츠**: Provider 네이티브(`image_url`) 방식과 표준 `content_blocks`(`url`) 방식 두 가지를 지원해요. 새 코드에서는 표준 방식을 권장해요
- **V1 import 경로**: `from langchain.messages import ...` (langchain_core 대신 langchain 사용)

## 다음 노트북 예고

다음 `04-StateGraph-Basics.ipynb`에서는 **StateGraph, State, Node, Edge, START/END**를 배워요. 이번 노트북에서 배운 메시지 타입들이 LangGraph의 상태(State)에 어떻게 저장되고 관리되는지, `add_messages` reducer가 메시지 ID를 어떻게 활용하는지 직접 확인할 수 있어요.